In [1]:
from keras.datasets import mnist
from sklearn.model_selection import KFold
import numpy as np

(Xtr, Ltr), (X_test, L_test)=mnist.load_data()

num_sample=500
Tr_set=Xtr[:num_sample,:,:]
Ltr_set=Ltr[:num_sample]


Tr_set=Tr_set.reshape(num_sample,Tr_set.shape[1]*Tr_set.shape[2]).astype(int)
Tr_set.shape

(500, 784)

In [2]:
def kNN_predict(testSet, trainingSet, trainingLabel, k_nearest):
    numerable = testSet.shape[0]
    Lpred = np.zeros(numerable, dtype=trainingLabel.dtype)
    
    # cast as int to not get bugs like before
    testSet = testSet.astype(int)
    trainingSet = trainingSet.astype(int)

    for i in range(numerable):
        distances = np.sqrt(np.sum((trainingSet - testSet[i, :]) ** 2, axis=1)) # L2
        nearest_indices = np.argsort(distances)[:k_nearest]
        nearest_labels = trainingLabel[nearest_indices]
        label_counts = np.bincount(nearest_labels, minlength=10)
        
        # break ties if needed.
        if np.count_nonzero(label_counts == label_counts.max()) > 1:
            winner = trainingLabel[nearest_indices[0]]
        else:
            winner = np.argmax(label_counts)
        Lpred[i] = winner
    
    return Lpred

In [3]:

def find_optimal_k(training_set, training_labels, k_values, num_folds=3):
    best_k = None
    best_accuracy = 0
    accuracies_for_k = {k: [] for k in k_values}
    
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=1)
    
    for k in k_values:
        fold_accuracies = []
        
        for train_index, val_index in kf.split(training_set):
            X_train_fold, X_val_fold = training_set[train_index], training_set[val_index]
            y_train_fold, y_val_fold = training_labels[train_index], training_labels[val_index]
            
            predictions = kNN_predict(X_val_fold, X_train_fold, y_train_fold, k)
            
            accuracy = np.mean(predictions == y_val_fold)
            fold_accuracies.append(accuracy)
        
        avg_accuracy = np.mean(fold_accuracies)
        accuracies_for_k[k].append(avg_accuracy)
        
        if avg_accuracy > best_accuracy:
            best_accuracy = avg_accuracy
            best_k = k
    
    return best_k, best_accuracy, accuracies_for_k

In [ ]:
k_values = list(range(1, 11)) # adjust k in the range 1 to 11 (more than enough for these tests)

# find the optimal k
best_k, best_accuracy, accuracies_for_k = find_optimal_k(Tr_set, Ltr_set, k_values)

print(f"optimal k: {best_k} with cross-validated accuracy: {best_accuracy} is the best")

# i just remove the np.float64 tag from the printouts so i can read it
accuracies_for_k_cleaned = {k: [float(acc) for acc in accuracies] for k, accuracies in accuracies_for_k.items()}
print("Accuracies for each k:", accuracies_for_k_cleaned)

optimal k: 4 with cross-validated accuracy: 0.8680229901642499
Accuracies for each k: {1: [0.8560589664045403], 2: [0.8560589664045403], 3: [0.8620229420676718], 4: [0.8680229901642499], 5: [0.8280427097612004], 6: [0.8379987013923959], 7: [0.8220306375201405], 8: [0.8199985571026621], 9: [0.8040064449414425], 10: [0.8059904287809921]}
